# Diagnóstico multiventana — horizonte de dos meses

Análisis exploratorio del Observatorio de Tikal y Flores, Petén. No publica resultados. Las predicciones de reparto que utilizan datos nacionales futuros son comparaciones ex-post.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.optimizacion_ventanas import evaluar_ventanas
DATA = ROOT / "data" / "processed"
print("Directorio:", DATA)

Directorio: C:\Users\Emadlg\Desktop\UVG\proyecto3_Data\data\processed


In [ ]:
mov = pd.read_csv(DATA / "movilidad_mensual_peten.csv")
mig = pd.read_csv(DATA / "migracion_nacional_mensual.csv")
ventanas, detalle, resumen = evaluar_ventanas(mov, mig, minimo_historia=2)
display(ventanas)
display(resumen)

,origen,mes_1,mes_2,grupos
0,2025-10-01,2025-11-01,2025-12-01,6
1,2026-04-01,2026-05-01,2026-06-01,6
2,2026-05-01,2026-06-01,2026-07-01,6
3,2026-06-01,2026-07-01,2026-08-01,6


,metodo,n,ventanas,MAE,requiere_nacional_futura,RMSE
0,persistencia,48,4,4444.315474,False,5936.485699
1,reparto_ultimo,24,2,5254.845254,True,6630.618952
2,media_movil,48,4,5480.496226,False,6808.567953
3,reparto_mediano,24,2,8253.281058,True,9256.588656


## Cobertura y ventanas
Una ventana requiere dos meses futuros consecutivos observados y dos meses anteriores consecutivos. No se rellenan los meses sin cobertura.

In [ ]:
if not ventanas.empty:
    display(ventanas.assign(origen=ventanas.origen.dt.strftime("%Y-%m"), mes_1=ventanas.mes_1.dt.strftime("%Y-%m"), mes_2=ventanas.mes_2.dt.strftime("%Y-%m")))
else:
    print("No hay ventanas válidas con los criterios actuales.")

,origen,mes_1,mes_2,grupos
0,2025-10,2025-11,2025-12,6
1,2026-04,2026-05,2026-06,6
2,2026-05,2026-06,2026-07,6
3,2026-06,2026-07,2026-08,6


## Comparación de métodos
Menor MAE y RMSE indican menor error, pero los métodos de reparto utilizan migración nacional futura observada como escenario ex-post.

In [ ]:
if not resumen.empty:
    fig = px.bar(resumen, x="metodo", y="MAE", title="MAE por método — ventanas disponibles")
    fig.show()
    display(resumen)
else:
    print("Sin evaluaciones suficientes.")

,metodo,n,ventanas,MAE,requiere_nacional_futura,RMSE
0,persistencia,48,4,4444.315474,False,5936.485699
1,reparto_ultimo,24,2,5254.845254,True,6630.618952
2,media_movil,48,4,5480.496226,False,6808.567953
3,reparto_mediano,24,2,8253.281058,True,9256.588656


In [ ]:
if not detalle.empty:
    errores = detalle.groupby(["origen","metodo"], as_index=False)["error_absoluto"].mean()
    errores["origen"] = errores["origen"].dt.strftime("%Y-%m")
    fig = px.bar(errores, x="origen", y="error_absoluto", color="metodo", barmode="group", title="Error absoluto por origen")
    fig.show()

In [ ]:
if not detalle.empty:
    display(detalle.groupby(["subdestino","segmento","metodo"], as_index=False)["error_absoluto"].mean().sort_values("error_absoluto").head(30))

,subdestino,segmento,metodo,error_absoluto
19,Mundo Maya Aeropuerto,mayor a 1 dia,reparto_ultimo,2063.461731
9,Isla De Flores,mayor a 1 dia,persistencia,3240.204513
3,El Remate Y Alrededores Lago,mayor a 1 dia,reparto_ultimo,3748.507102
8,Isla De Flores,mayor a 1 dia,media_movil,3915.936888
17,Mundo Maya Aeropuerto,mayor a 1 dia,persistencia,4103.090009
1,El Remate Y Alrededores Lago,mayor a 1 dia,persistencia,4135.406877
16,Mundo Maya Aeropuerto,mayor a 1 dia,media_movil,4298.438553
11,Isla De Flores,mayor a 1 dia,reparto_ultimo,4414.157489
13,Isla De Flores,menor a 1 dia,persistencia,4435.259557
0,El Remate Y Alrededores Lago,mayor a 1 dia,media_movil,4606.369301


## Conclusión metodológica
Esta evaluación retrospectiva sirve para detectar estabilidad del desempeño. No demuestra por sí sola cuál modelo habría sido elegible antes de enero de 2026. Los meses posteriores a esa fecha solo sirven como evaluación ex-post. No modificar los umbrales de MLOps para aprobar un candidato rechazado.

## Comparación final sobre ventanas comunes
Se separan métodos operativos de métodos de reparto que usan migración nacional futura observada (ex-post). Las métricas de cada tabla se calculan sobre las mismas unidades de evaluación. Esta selección retrospectiva no constituye aprobación para publicación.

In [ ]:
from src.optimizacion_ventanas import comparaciones_finales
comparaciones = comparaciones_finales(detalle)
for nombre, (pares, tabla) in comparaciones.items():
    print(f"Comparación: {nombre}; registros emparejados: {len(pares)}")
    display(tabla)
    if not tabla.empty:
        fig = px.bar(tabla, x="metodo", y="MAE", title=f"MAE — comparación {nombre} (mismas ventanas)")
        fig.show()


Comparación: operativa; registros emparejados: 96


,metodo,n,ventanas,MAE,RMSE,requiere_nacional_futura
0,persistencia,48,4,4444.315474,5936.485699,False
1,media_movil,48,4,5480.496226,6808.567953,False


Comparación: expost; registros emparejados: 96


,metodo,n,ventanas,MAE,RMSE,requiere_nacional_futura
0,reparto_ultimo,24,2,5254.845254,6630.618952,True
1,persistencia,24,2,6810.547355,8036.210870,False
2,media_movil,24,2,7282.613933,8655.656462,False
3,reparto_mediano,24,2,8253.281058,9256.588656,True


In [ ]:
operativa = comparaciones["operativa"][1]
if not operativa.empty:
    print("Candidato exploratorio:", operativa.sort_values("MAE").iloc[0]["metodo"])
    print("No actualizar predicciones.py ni los umbrales de MLflow sin evaluación adicional.")


Candidato exploratorio: persistencia
No actualizar predicciones.py ni los umbrales de MLflow sin evaluación adicional.
